# M-Lab India — Stage 1: visual inspection, coverage, and outlier removal

This notebook pairs row-level audits with charts. Access-network exclusions use reviewed ASN rules;
measurement values and IP geolocation provide context, not proof of access type. It (1) checks
coverage and missingness, (2) shows geo-spatial coverage, (3) applies exclusions and **removes**
two exclusion categories — networks identified by exact ASN review and an impossible sentinel
value — and (4) shows a third category (extreme-but-plausible statistical tails) as
**review-only**; the decision (keep) is finalized in Section 8a. A clean dataset is exported at the end.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'mlab_analysis':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DATA_DIR = Path(os.environ.get('MLAB_RAW_DATA_DIR', PROJECT_ROOT / 'mlab_data' / 'combined' / '2026-06-18_to_2026-11-04'))
DOWNLOAD_PATH = RAW_DATA_DIR / 'download.csv'
UPLOAD_PATH = RAW_DATA_DIR / 'upload.csv'
OUTPUT_DIR = Path(os.environ.get('MLAB_OUTPUT_DIR', PROJECT_ROOT / 'outputs' / 'analysis' / '2026-06-18_to_2026-11-04'))
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def figure_path(section, filename):
    path = FIGURE_DIR / section / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    return path

BLUE, ORANGE, RED, GREY, GREEN = '#4C78A8', '#F58518', '#E45756', '#B0B0B0', '#54A24B'

STRING_COLUMNS = ['measurement_id', 'date', 'test_time_utc', 'country_code',
                  'state_code', 'state', 'city', 'postal_code', 'client_asn',
                  'client_as_name', 'client_network_cidr', 'congestion_control']

def load_csv(path):
    columns = pd.read_csv(path, nrows=0).columns
    dtypes = {c: 'string' for c in STRING_COLUMNS if c in columns}
    return pd.read_csv(path, dtype=dtypes, low_memory=False)

download_raw = load_csv(DOWNLOAD_PATH)
upload_raw = load_csv(UPLOAD_PATH)

DOWNLOAD_FEATURES = ['download_mbps', 'min_rtt_ms', 'loss_rate']
UPLOAD_FEATURES = ['upload_mbps', 'min_rtt_ms']

upload_loss_columns = [c for c in ['loss_rate_raw', 'loss_percent_raw'] if c in upload_raw.columns]
download = download_raw.copy()
upload = upload_raw.drop(columns=upload_loss_columns).copy()

print(f'download: {download.shape}   upload: {upload.shape}')

## 1. Data coverage

Bars show measurements per calendar day against the full observed window. A day with zero rows,
or a handful of days holding almost all the rows, is a sampling-query red flag — not present here.

In [ ]:
all_dates = pd.concat([pd.to_datetime(download['date']), pd.to_datetime(upload['date'])])
expected_dates = pd.date_range(all_dates.min(), all_dates.max(), freq='D')

coverage_parts = []
for name, df in [('download', download), ('upload', upload)]:
    counts = pd.to_datetime(df['date']).value_counts().reindex(expected_dates, fill_value=0).sort_index()
    coverage_parts.append(pd.DataFrame({'dataset': name, 'date': counts.index,
                                        'row_count': counts.values, 'date_present': counts.values > 0}))
coverage = pd.concat(coverage_parts, ignore_index=True)
coverage.assign(date=coverage['date'].dt.strftime('%Y-%m-%d')).to_csv(OUTPUT_DIR / 'date_coverage.csv', index=False)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
for ax, (name, part), color in zip(axes, coverage.groupby('dataset', sort=False), [BLUE, ORANGE]):
    observed = int(part['date_present'].sum())
    ax.bar(part['date'], part['row_count'], width=.85, color=color)
    ax.set_title(f'{name.title()}: {observed}/{len(expected_dates)} expected days present  '
                 f'({part["row_count"].sum():,} rows)')
    ax.set_ylabel('Rows/day')
    ax.grid(axis='y', alpha=.25)
axes[-1].set_xlabel(f'{expected_dates.min():%Y-%m-%d} to {expected_dates.max():%Y-%m-%d}')
fig.tight_layout(); fig.savefig(figure_path('01_data_quality', 'date_coverage.png'), dpi=180); plt.show()

## 2. Missing values per variable

Only columns with any missingness are shown. Core clustering features
(`download_mbps` / `upload_mbps`, `min_rtt_ms`, `loss_rate`) are checked separately below and are
not part of this chart when fully populated — missingness here is confined to optional geo/ISP
metadata, so it does not drive any row removal.

In [ ]:
def missingness(df, dataset):
    out = pd.DataFrame({'column': df.columns, 'dtype': df.dtypes.astype(str).values,
                        'missing_count': df.isna().sum().values})
    out.insert(0, 'dataset', dataset)
    out['missing_percent'] = 100 * out['missing_count'] / len(df)
    return out

schema_missingness = pd.concat([missingness(download_raw, 'download'),
                                missingness(upload_raw, 'upload')], ignore_index=True)
schema_missingness.to_csv(OUTPUT_DIR / 'schema_missingness.csv', index=False)

missing_plot = schema_missingness.loc[schema_missingness['missing_percent'].gt(0)].copy()
missing_plot['label'] = missing_plot['dataset'] + ': ' + missing_plot['column']
missing_plot = missing_plot.sort_values('missing_percent')

fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(missing_plot))))
colors = missing_plot['dataset'].map({'download': BLUE, 'upload': ORANGE})
ax.barh(missing_plot['label'], missing_plot['missing_percent'], color=colors)
for y, pct in enumerate(missing_plot['missing_percent']):
    ax.text(pct + 0.15, y, f'{pct:.2f}%', va='center', fontsize=8)
feature_cols = {'download_mbps', 'upload_mbps', 'min_rtt_ms', 'loss_rate'}
flagged_features = missing_plot.loc[missing_plot['column'].isin(feature_cols)]
ax.set_title('Columns with any missing values (clustering features shown in red if present above)')
ax.set_xlabel('Missing rows (%)')
ax.grid(axis='x', alpha=.25)
handles = [Line2D([0], [0], color=BLUE, lw=6, label='download'), Line2D([0], [0], color=ORANGE, lw=6, label='upload')]
ax.legend(handles=handles, loc='lower right')
fig.tight_layout(); fig.savefig(figure_path('01_data_quality', 'missingness_by_column.png'), dpi=200, bbox_inches='tight'); plt.show()

print('Clustering-feature columns with missing values:', flagged_features['label'].tolist() or 'none')

## 3. Domain-invalid values (impossible by definition)

A row is domain-invalid only if a clustering feature is missing/non-finite, throughput or RTT is
non-positive, download loss falls outside `[0, 1]`, the measurement ID is duplicated, or a value is
a known **sentinel/overflow artifact** rather than a physical measurement. These are removed
outright — they cannot represent a real network condition, so "retain extreme values" does not
apply to them.

In [ ]:
UINT32_MS_SENTINEL = (2**32 - 1) / 1000.0  # 4294967.295 -- classic unsigned-32-bit overflow/error code

def domain_validity(df, dataset, features, throughput, loss=None):
    numeric = df[features].apply(pd.to_numeric, errors='coerce')
    reasons = {
        'missing_or_nonnumeric_feature': numeric.isna().any(axis=1),
        'nonfinite_feature': ~np.isfinite(numeric).all(axis=1),
        'nonpositive_throughput': numeric[throughput] <= 0,
        'nonpositive_rtt': numeric['min_rtt_ms'] <= 0,
        'rtt_uint32_sentinel': np.isclose(numeric['min_rtt_ms'], UINT32_MS_SENTINEL, atol=1.0),
        'duplicate_measurement_id': df['measurement_id'].duplicated(keep=False),
        'loss_outside_0_1': pd.Series(False, index=df.index),
    }
    if loss:
        reasons['loss_outside_0_1'] = ~numeric[loss].between(0, 1)
    result = pd.DataFrame(reasons)
    result.insert(0, 'dataset', dataset)
    result['domain_invalid'] = result[list(reasons)].any(axis=1)
    result.index = df.index
    return result

download_validity = domain_validity(download, 'download', DOWNLOAD_FEATURES, 'download_mbps', 'loss_rate')
upload_validity = domain_validity(upload, 'upload', UPLOAD_FEATURES, 'upload_mbps')

validity_counts = pd.concat([download_validity, upload_validity]).drop(columns='domain_invalid') \
    .groupby('dataset').sum().T
validity_counts.to_csv(OUTPUT_DIR / 'domain_validity_counts.csv')

fig, ax = plt.subplots(figsize=(9, 4))
validity_counts.plot.barh(ax=ax, color=[BLUE, ORANGE])
ax.set(title='Domain-invalid reason counts (0 bars omitted from the story, kept for audit)',
       xlabel='Rows flagged')
ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(figure_path('01_data_quality', 'domain_invalid_reasons.png'), dpi=200, bbox_inches='tight'); plt.show()

# sanity checks that are invariants, not removal decisions
assert np.isclose(download['loss_percent'], 100 * download['loss_rate']).all()
print('loss_percent == 100 * loss_rate for every download row: OK')

sentinel_rows = pd.concat([
    download.loc[download_validity['rtt_uint32_sentinel']].assign(dataset='download'),
    upload.loc[upload_validity['rtt_uint32_sentinel']].assign(dataset='upload'),
])
print(f'Sentinel RTT rows found: {len(sentinel_rows)} (uint32 overflow constant = {UINT32_MS_SENTINEL:,.3f} ms)')
display(sentinel_rows[['dataset', 'measurement_id', 'date', 'min_rtt_ms', 'client_asn', 'client_as_name']])

### Evidence: the sentinel RTT is detached from the real distribution

`min_rtt_ms = 4294967.295` is **exactly** `(2**32 - 1) / 1000` — the classic 32-bit unsigned integer
overflow/"no value" sentinel, divided by 1000 as if it were milliseconds. No real round-trip time is
~50 days long. The chart below shows it sitting ~1,400x past the next-highest observed RTT, isolated
from every other point.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ordered = np.sort(upload['min_rtt_ms'].dropna().to_numpy())
ax.plot(np.arange(1, len(ordered) + 1), ordered, color=BLUE, lw=1.6)
sentinel_mask = np.isclose(ordered, UINT32_MS_SENTINEL, atol=1.0)
ax.scatter(np.flatnonzero(sentinel_mask) + 1, ordered[sentinel_mask], color=RED, s=90, zorder=5,
           label='Sentinel overflow value')
ax.set_yscale('log')
ax.set(title='Upload min_rtt_ms, sorted (log y-axis) — one point is ~1,400x the next-highest',
       xlabel='Rank', ylabel='min_rtt_ms (log scale)')
ax.grid(alpha=.25, which='both'); ax.legend()
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'sentinel_rtt_evidence.png'), dpi=200, bbox_inches='tight'); plt.show()

## 4. Distributions & ECDFs (log-scale, full data)

Log-scaled x-axes expose the poor-network tail without letting the high end hide it. Loss stays on
its natural linear scale. Shown here on the still-uncleaned data for context; cleaned-data
comparisons appear in the outlier sections below.

In [ ]:
def smooth_histogram(density, sigma=1.5):
    radius = max(2, int(4 * sigma))
    x = np.arange(-radius, radius + 1)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)
    kernel /= kernel.sum()
    return np.convolve(density, kernel, mode='same')

def distribution_ecdf(df, dataset, features):
    fig, axes = plt.subplots(len(features), 2, figsize=(12, 3.5 * len(features)))
    axes = np.atleast_2d(axes)
    for row, feature in enumerate(features):
        values = df[feature].dropna().to_numpy()
        is_loss = 'loss' in feature
        bins = np.linspace(0, values.max(), 50) if is_loss else np.geomspace(values.min(), values.max(), 50)
        density, edges, _ = axes[row, 0].hist(values, bins=bins, density=True, color=BLUE, alpha=.55, label='Histogram')
        centers = (edges[:-1] + edges[1:]) / 2 if is_loss else np.sqrt(edges[:-1] * edges[1:])
        if not is_loss:
            axes[row, 0].plot(centers, smooth_histogram(density), color=RED, lw=2.2, label='Smoothed density')
        ordered = np.sort(values)
        axes[row, 1].plot(ordered, np.arange(1, len(ordered)+1)/len(ordered), color=ORANGE)
        if not is_loss:
            axes[row, 0].set_xscale('log'); axes[row, 1].set_xscale('log')
        p01, median, p99 = np.quantile(values, [.01, .5, .99])
        axes[row, 0].axvline(median, color='black', ls='--', lw=1.2, label='Median')
        axes[row, 1].axvline(median, color='black', ls='--', lw=1.2)
        axes[row, 0].set(title=f'{dataset.title()}: {feature} distribution', xlabel=feature, ylabel='Density')
        axes[row, 1].set(title=f'{dataset.title()}: {feature} ECDF', xlabel=feature, ylabel='Cumulative probability', ylim=(0, 1.01))
        axes[row, 0].text(.02, .95, f'P1={p01:.3g}  Median={median:.3g}  P99={p99:.3g}', transform=axes[row, 0].transAxes, va='top', fontsize=8)
        axes[row, 0].legend(fontsize=8); axes[row, 0].grid(alpha=.25); axes[row, 1].grid(alpha=.25)
    fig.tight_layout(); plt.show()

distribution_ecdf(download, 'download', DOWNLOAD_FEATURES)
distribution_ecdf(upload, 'upload', UPLOAD_FEATURES)

## 5. Geo-spatial analysis

IP-geolocation coordinates, treated as approximate centroids, not verified addresses. Density maps
show where measurements concentrate; the throughput-coloured maps set up the data-centre evidence
in the next section — data-centre points show up as small, extremely fast, geographically sparse
clusters distinct from the broad residential-access footprint.

In [ ]:
india_bbox = lambda df: df['latitude'].between(6, 38) & df['longitude'].between(68, 98)
geo_rows = []
for name, df in [('download', download), ('upload', upload)]:
    geo_rows.append({'dataset': name, 'states': df['state'].nunique(), 'cities': df['city'].nunique(),
                      'coordinate_pairs': len(df[['latitude', 'longitude']].drop_duplicates()),
                      'outside_india_bbox': int((~india_bbox(df)).sum())})
geo_summary = pd.DataFrame(geo_rows)
geo_summary.to_csv(OUTPUT_DIR / 'geographic_coverage_summary.csv', index=False)
display(geo_summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, (name, df) in zip(axes, [('download', download), ('upload', upload)]):
    mesh = ax.hexbin(df['longitude'], df['latitude'], gridsize=38, mincnt=1, bins='log', cmap='viridis')
    ax.set(xlim=(67, 99), ylim=(6, 38), title=f'{name.title()}: measurement density', xlabel='Longitude', ylabel='Latitude')
    ax.grid(alpha=.15); fig.colorbar(mesh, ax=ax, label='log10(rows + 1)')
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, (name, df, col) in zip(axes, [('download', download, 'download_mbps'), ('upload', upload, 'upload_mbps')]):
    sc = ax.scatter(df['longitude'], df['latitude'], c=df[col], cmap='inferno',
                     norm=plt.matplotlib.colors.LogNorm(), s=4, alpha=.5)
    ax.set(xlim=(67, 99), ylim=(6, 38), title=f'{name.title()}: throughput by location', xlabel='Longitude', ylabel='Latitude')
    ax.grid(alpha=.15); fig.colorbar(sc, ax=ax, label=f'{col} (log scale)')
fig.tight_layout(); plt.show()

top_states = download['state'].value_counts().head(12).index
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharey=True)
for ax, (name, df, col) in zip(axes, [('download', download, 'download_mbps'), ('upload', upload, 'upload_mbps')]):
    med = df[df['state'].isin(top_states)].groupby('state')[col].median().sort_values()
    ax.barh(med.index, med.values, color=BLUE if name == 'download' else ORANGE)
    ax.set(title=f'{name.title()}: median {col} by top-12 state', xlabel=f'Median {col}')
    ax.grid(axis='x', alpha=.25)
fig.tight_layout(); plt.show()

## 6. Outlier evidence A — data-centre / enterprise / institutional / non-access networks (removed)

Candidate status requires an **exact ASN match** to a reviewed list of cloud, hosting, CDN,
data-centre, proxy/security, enterprise, government/institutional or academic-network operators —
never a name-substring guess, since Indian access ISPs legitimately use words like "cloud",
"technologies" or "networks" in their names. This is the "rich enterprise/data-centre" outlier you
flagged: these are cloud VM/CDN egress links or institutional campus/backbone networks, not
residential or mobile access-network conditions, so they do not belong in an access-network study
population. Round 2 (below) adds institutions found by manually reviewing every ASN behind a
high-throughput row in the clean data, cross-checked against each ASN's actual traffic shape
(RTT/geographic spread) so that ordinary ISPs with enterprise-sounding names — e.g. India's
National Internet Backbone (BSNL) — are not swept in by mistake.

In [ ]:
NON_ACCESS_RULES = {
    36183: ('cdn_cloud', 'Akamai'), 12222: ('cdn_cloud', 'Akamai'),
    16509: ('cloud_hosting', 'Amazon'), 13335: ('cdn_proxy', 'Cloudflare'),
    18229: ('data_center', 'CtrlS Datacenters'), 396982: ('cloud_hosting', 'Google'),
    36692: ('proxy_security', 'OpenDNS'), 8075: ('cloud_enterprise', 'Microsoft'),
    31898: ('cloud_hosting', 'Oracle'), 17439: ('data_center', 'Netmagic'),
    14061: ('cloud_hosting', 'DigitalOcean'), 16276: ('cloud_hosting', 'OVH'),
    63949: ('cloud_hosting', 'Linode'), 54113: ('cdn_cloud', 'Fastly'),
    132420: ('cloud_hosting', 'E2E Networks'), 132335: ('cloud_hosting', 'LeapSwitch'),
    134926: ('cloud_hosting', 'Micro Hosting'), 714: ('cloud_enterprise', 'Apple'),
    44444: ('proxy_security', 'Forcepoint'), 13150: ('proxy_security', 'Cato Networks'),
    137922: ('proxy_security', 'iboss'), 53813: ('proxy_security', 'Zscaler'),
    40934: ('proxy_security', 'Fortinet'), 27385: ('proxy_security', 'Qualys'),
    401624: ('enterprise_colocation', 'Omnissa / colocated prefix'),
    # Round 2: found by reviewing every ASN behind a high-throughput row in the clean data.
    4758: ('government_institutional', 'National Informatics Centre'),
    11911: ('financial_enterprise', 'Bank of New York Mellon'),
    45815: ('cloud_hosting', 'ESDS Software Solution (Indian data centre)'),
    254: ('enterprise', 'The Wollongong Group'),
    109: ('cloud_enterprise', 'Cisco Systems'),
    16999: ('enterprise', 'World Fuel Services Corporation'),
    132215: ('government_institutional', 'Power Grid Corporation of India'),
    55824: ('academic_research_network', 'NKN Core Network (National Knowledge Network)'),
    132780: ('academic_institution', 'IIT Delhi'),
    55479: ('academic_institution', 'IIT Kanpur'),
    131265: ('enterprise', 'TCS eServe Limited'),
    137590: ('government_institutional', 'UA E-Governance PMU'),
    135836: ('unresolved_extreme_speed', 'Advance Multisystem Broadband (2838 Mbps @ 1.2ms, single row)'),
    # Round 3: found while building the connection-type (mobile vs. broadband) breakdown below.
    55256: ('proxy_security', 'Netskope Inc (cloud SASE/proxy, median RTT 1.68ms across 3 metros)'),
    394089: ('unresolved_identity', 'Oahu Publications (ASN identity does not match traffic origin; '
             'removed by explicit decision despite access-like RTT/throughput)'),
    # Round 4: additional named institutional, corporate, data-centre, and Wi-Fi networks
    # found in the combined June-November data; exact ASN matches only.
    131226: ('academic_institution', 'Indian Institute of Technology Roorkee'),
    59163: ('academic_institution', 'GLA University'),
    132785: ('academic_institution', 'Shiv Nadar University'),
    137282: ('academic_institution', 'KIIT University'),
    131473: ('academic_institution', 'SRM University'),
    132749: ('academic_institution', 'Indraprastha Institute of Information Technology, Delhi'),
    138155: ('academic_institution', 'Jawaharlal Nehru University'),
    56166: ('academic_institution', 'IISER Bhopal Campus'),
    133552: ('academic_institution', 'B.M.S College of Engineering'),
    136470: ('academic_institution', 'B.M.S College of Engineering'),
    137956: ('academic_institution', 'Indian Institute of Technology Ropar'),
    132524: ('academic_institution', 'Tata Institute of Fundamental Research'),
    134023: ('academic_institution', 'Aligarh Muslim University'),
    133273: ('academic_institution', 'Tata Institute of Social Sciences'),
    134854: ('enterprise', 'Robert Bosch Engineering and Business Solutions'),
    132401: ('enterprise', 'Comcast India Engineering Center'),
    132075: ('enterprise', 'Tech Mahindra Business Services'),
    63935: ('enterprise', 'Mu Sigma Business Solution'),
    4983: ('enterprise', 'Intel Corporation'),
    3598: ('enterprise', 'Microsoft Corporation'),
    136590: ('enterprise', 'OpenText Corporation'),
    135133: ('data_center', 'PI Data Centers Private Limited'),
    7633: ('government_institutional', 'Software Technology Parks of India'),
    9430: ('government_institutional', 'Software Technology Parks of India, Block IV'),
    24515: ('government_institutional', 'Software Technology Parks of India, Jaipur'),
    45596: ('government_institutional', 'Centre for Railway Information Systems'),
    135862: ('wifi_service_name', 'IRG WIFI Service'),
}
ROUND_4_ASNS = [131226, 59163, 132785, 137282, 131473, 132749, 138155, 56166,
                133552, 136470, 137956, 132524, 134023, 133273, 134854, 132401,
                132075, 63935, 4983, 3598, 136590, 135133, 7633, 9430, 24515,
                45596, 135862]
# Checked and explicitly NOT added, despite enterprise/institutional-sounding names:
# AS9829 National Internet Backbone (BSNL) -- 1,321 rows, median RTT 22.6ms, spread across
#   32 states/191 cities: ordinary broadband customers, not a data-centre signature.
# AS24186 RailTel Corporation -- 319 rows, median RTT 24.6ms, 12 states/22 cities: same pattern.
# AS132559 Gatik Business Solutions, AS18002 (generic ASN name): single-city small ISPs, normal RTT.
# AS132570/AS132566/AS137157 "Skynet ..." -- 1 row each, RTT 21-48ms, ordinary ISP numbers;
#   "Skynet" is just a brand name here, not a data-centre/military signature.
# AS132972 "171 Incorporation" -- 11 rows, all Bhagalpur (Bihar), RTT 56-114ms: a small regional
#   ISP in an underserved area, exactly the population this dataset should keep.
rules = pd.DataFrame([{'client_asn_numeric': asn, 'network_category': v[0], 'review_identity': v[1]}
                      for asn, v in NON_ACCESS_RULES.items()])
rules.to_csv(OUTPUT_DIR / 'non_access_asn_rules.csv', index=False)

additional_rows = []
for dataset_name, frame in [('download', download), ('upload', upload)]:
    asn_values = pd.to_numeric(frame['client_asn'], errors='coerce')
    counts = asn_values.value_counts()
    for asn in ROUND_4_ASNS:
        category, identity = NON_ACCESS_RULES[asn]
        additional_rows.append({'client_asn_numeric': asn, 'review_identity': identity,
                                'network_category': category, 'dataset': dataset_name,
                                'rows_removed': int(counts.get(asn, 0))})
pd.DataFrame(additional_rows).to_csv(OUTPUT_DIR / 'additional_non_access_asn_review.csv', index=False)

def flag_non_access(df, dataset):
    audited = df.copy()
    audited['client_asn_numeric'] = pd.to_numeric(audited['client_asn'], errors='coerce').astype('Int64')
    audited = audited.merge(rules, on='client_asn_numeric', how='left')
    audited['non_access_candidate'] = audited['network_category'].notna()
    audited.insert(0, 'dataset', dataset)
    audited.index = df.index
    return audited

download_audit = flag_non_access(download, 'download')
upload_audit = flag_non_access(upload, 'upload')
non_access_download = download_audit['non_access_candidate']
non_access_upload = upload_audit['non_access_candidate']

print(f'Exact-ASN policy rows flagged — download: {non_access_download.sum()} '
      f'({100*non_access_download.mean():.2f}%)   upload: {non_access_upload.sum()} '
      f'({100*non_access_upload.mean():.2f}%)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, audited, throughput in [(axes[0], download_audit, 'download_mbps'), (axes[1], upload_audit, 'upload_mbps')]:
    normal = audited.loc[~audited['non_access_candidate']]
    flagged = audited.loc[audited['non_access_candidate']]
    ax.scatter(normal[throughput], normal['min_rtt_ms'], s=4, alpha=.12, color=GREY, label=f'Access network (n={len(normal):,})')
    ax.scatter(flagged[throughput], flagged['min_rtt_ms'], s=22, alpha=.85, color=RED, label=f'Exact-ASN policy rows (n={len(flagged):,})')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set(title=f'{audited["dataset"].iloc[0].title()}: throughput vs RTT', xlabel=throughput, ylabel='min_rtt_ms')
    ax.grid(alpha=.18, which='both'); ax.legend(fontsize=8)
fig.suptitle('Exact-ASN policy rows vs remaining observations (measurements do not prove access type)')
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'non_access_evidence_scatter.png'), dpi=200, bbox_inches='tight'); plt.show()

asn_counts = pd.concat([
    download_audit.loc[non_access_download, ['client_as_name']].assign(dataset='download'),
    upload_audit.loc[non_access_upload, ['client_as_name']].assign(dataset='upload'),
]).groupby(['client_as_name', 'dataset']).size().rename('rows').reset_index()
top_asns = asn_counts.groupby('client_as_name')['rows'].sum().sort_values().tail(15).index
pivot = asn_counts[asn_counts['client_as_name'].isin(top_asns)].pivot(index='client_as_name', columns='dataset', values='rows').fillna(0)
pivot = pivot.loc[pivot.sum(axis=1).sort_values().index]
ax = pivot.plot.barh(figsize=(10, 6), color=[BLUE, ORANGE])
ax.set(title='Top flagged organizations by row count', xlabel='Rows removed')
ax.grid(axis='x', alpha=.25)
plt.tight_layout(); plt.savefig(figure_path('02_cleaning_evidence', 'non_access_top_asns.png'), dpi=200, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, audited in zip(axes, [download_audit, upload_audit]):
    ax.scatter(audited['longitude'], audited['latitude'], s=3, alpha=.08, color=GREY, label='Access network')
    flagged = audited[audited['non_access_candidate']]
    ax.scatter(flagged['longitude'], flagged['latitude'], s=18, alpha=.75, color=RED, label='Exact-ASN policy rows')
    ax.set(xlim=(67, 99), ylim=(6, 38), title=audited['dataset'].iloc[0].title(), xlabel='Longitude', ylabel='Latitude')
    ax.grid(alpha=.15); ax.legend(fontsize=8)
fig.suptitle('Flagged locations are IP geolocation, not facility proof')
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, audited, throughput, label in [(axes[0], download_audit, 'download_mbps', 'Download Mbps'),
                                        (axes[1], upload_audit, 'upload_mbps', 'Upload Mbps')]:
    for frame, name, color in [(audited, 'Before removal (all)', BLUE),
                                (audited.loc[~audited['non_access_candidate']], 'After removal', GREEN)]:
        ordered = np.sort(frame[throughput])
        ax.plot(ordered, np.arange(1, len(ordered)+1)/len(ordered), label=name, color=color, lw=2)
    ax.set_xscale('log'); ax.set(title=f'{label}: effect of removing data-centre rows', xlabel=label, ylabel='ECDF', ylim=(0, 1.01))
    ax.grid(alpha=.2); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'non_access_before_after_ecdf.png'), dpi=200); plt.show()

### 6a. Every manually-removed category, not just the top names

The chart above only shows the top 15 organizations *by row count* — small entries (1-2 rows each)
don't show up there even though they were just as deliberately reviewed and removed. This table
rolls all ~40 manually-curated ASNs up by category, so nothing is left implicit.

In [ ]:
category_rollup = []
for cat, group in rules.groupby('network_category'):
    asns = group['client_asn_numeric'].tolist()
    dl_rows = int(download_audit['client_asn_numeric'].isin(asns).sum())
    ul_rows = int(upload_audit['client_asn_numeric'].isin(asns).sum())
    examples = ', '.join(group['review_identity'].str.split(' \(').str[0].head(2))
    category_rollup.append({'Category': cat.replace('_', ' ').title(), 'ASNs': len(asns),
                             'Example organizations': examples, 'Download rows': dl_rows,
                             'Upload rows': ul_rows, 'Total rows removed': dl_rows + ul_rows})
category_rollup = pd.DataFrame(category_rollup).sort_values('Total rows removed', ascending=False).reset_index(drop=True)
category_rollup.to_csv(OUTPUT_DIR / 'non_access_category_rollup.csv', index=False)
display(category_rollup)

fig, ax = plt.subplots(figsize=(13, 0.55 * len(category_rollup) + 1))
ax.axis('off')
tbl = ax.table(cellText=category_rollup.values, colLabels=category_rollup.columns, cellLoc='center', loc='center',
               colWidths=[.18, .06, .42, .11, .11, .12])
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 2.0)
for j in range(len(category_rollup.columns)):
    tbl[(0, j)].set_facecolor(BLUE); tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for i in range(1, len(category_rollup) + 1):
    for j in range(len(category_rollup.columns)):
        tbl[(i, j)].set_facecolor('#FDECEA' if i % 2 else '#FBF3F2')
    tbl[(i, 0)].set_text_props(ha='left')
    tbl[(i, 2)].set_text_props(ha='left')
fig.suptitle('Manually-removed ASNs by category (all rounds, not just the top names)', y=.99, fontsize=12)
fig.tight_layout(); plt.show()

**Reasoning:** these rows come from cloud VM egress, CDN edge nodes, or enterprise
proxy/security gateways — infrastructure with data-centre-grade uplinks, not a phone or home
router on an Indian ISP. Mixing them in would bias throughput/RTT clusters toward speeds no
real access-network user experiences. Per your instruction, these are removed from the clean
dataset built in Section 9.

## 7. Outlier evidence B — impossible sentinel value (removed)

Already isolated in Section 3: the single upload row with `min_rtt_ms` exactly matching the
32-bit overflow constant is a parser/encoding artifact, not a measurement. Removed unconditionally.

In [ ]:
print(f'Rows removed for uint32 RTT sentinel: {len(sentinel_rows)}')
display(sentinel_rows[['dataset', 'measurement_id', 'date', 'min_rtt_ms', 'client_asn', 'client_as_name', 'state', 'city']])

## 8. Outlier evidence C — statistical tail (review only — **your call**)

After removing the two categories above, a robust modified z-score on `log1p`-transformed features
(median/MAD based, threshold `|z| > 5`) flags points that are numerically extreme but not
impossible. Red points below are **retained for now** — they may be genuine severe-network
conditions (very poor rural/mobile links) rather than errors. I have **not** removed them.

In [ ]:
clean_stage1_download = download.loc[~download_validity['domain_invalid'] & ~non_access_download].copy()
clean_stage1_upload = upload.loc[~upload_validity['domain_invalid'] & ~non_access_upload].copy()

def modified_z(series, multiplier=1):
    transformed = np.log1p(multiplier * series)
    median = transformed.median()
    mad = (transformed - median).abs().median()
    return .67448975 * (transformed - median) / mad

def robust_joint_review(df, transforms, review_fraction=.001):
    components = {}
    for feature, multiplier in transforms.items():
        values = np.log1p(multiplier * df[feature])
        median = values.median()
        mad = (values - median).abs().median()
        components[feature] = .67448975 * (values - median) / mad
    components = pd.DataFrame(components, index=df.index)
    score = np.sqrt((components ** 2).sum(axis=1))
    cutoff = score.quantile(1 - review_fraction)
    return score, score >= cutoff, cutoff

download_joint_score, download_joint_flag, download_joint_cutoff = robust_joint_review(
    clean_stage1_download, {'download_mbps': 1, 'min_rtt_ms': 1, 'loss_rate': 100})
upload_joint_score, upload_joint_flag, upload_joint_cutoff = robust_joint_review(
    clean_stage1_upload, {'upload_mbps': 1, 'min_rtt_ms': 1})

flag_counts = pd.DataFrame([
    {'dataset': 'download', 'rows': len(clean_stage1_download), 'joint_review_rows': int(download_joint_flag.sum())},
    {'dataset': 'upload', 'rows': len(clean_stage1_upload), 'joint_review_rows': int(upload_joint_flag.sum())},
]).set_index('dataset')
flag_counts['review_percent'] = 100 * flag_counts['joint_review_rows'] / flag_counts['rows']
flag_counts.to_csv(OUTPUT_DIR / 'statistical_tail_review_counts.csv')
display(flag_counts.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, df, throughput, flag, title in [
    (axes[0], clean_stage1_download, 'download_mbps', download_joint_flag, 'Download (cleaned by ASN policy + validity checks)'),
    (axes[1], clean_stage1_upload, 'upload_mbps', upload_joint_flag, 'Upload (cleaned by ASN policy + validity checks)')]:
    density = ax.hexbin(df[throughput], df['min_rtt_ms'], xscale='log', yscale='log',
                        gridsize=55, mincnt=1, bins='log', cmap='Blues')
    marked = df.loc[flag]
    ax.scatter(marked[throughput], marked['min_rtt_ms'], facecolors='none', edgecolors=RED,
               linewidths=1.2, s=42, label=f'Top 0.1% joint-robust-distance review ({len(marked)})')
    ax.set(title=title, xlabel=throughput, ylabel='min_rtt_ms', xscale='log', yscale='log')
    ax.grid(alpha=.16, which='both'); ax.legend(loc='upper right', fontsize=8)
    fig.colorbar(density, ax=ax, label='log10(rows + 1)')
fig.suptitle('Statistical-tail candidates: highlighted rows are RETAINED, not removed', y=1.02)
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'statistical_tail_review.png'), dpi=200, bbox_inches='tight'); plt.show()

statistical_flags = pd.concat([
    clean_stage1_download.loc[download_joint_flag, ['measurement_id', 'date', 'state', 'city', 'client_asn', 'client_as_name']].assign(dataset='download'),
    clean_stage1_upload.loc[upload_joint_flag, ['measurement_id', 'date', 'state', 'city', 'client_asn', 'client_as_name']].assign(dataset='upload'),
], ignore_index=True)
statistical_flags.to_csv(OUTPUT_DIR / 'statistical_tail_flags.csv', index=False)

### 8a. Tail deep-dive — is this noise or real severe-network conditions?

Three checks before you decide: **(1) diversity** — is the tail spread across many ASNs/states/dates
(consistent with many different bad links) or concentrated on one ASN/date (consistent with a bug)?
**(2) shape** — how extreme are these points relative to the rest of the cleaned data? **(3)
corroboration** — for download, does high RTT come with high loss too, which a real congested/poor
path would show, or is it an isolated single-feature spike?

In [ ]:
flagged_download = clean_stage1_download.loc[download_joint_flag]
flagged_upload = clean_stage1_upload.loc[upload_joint_flag]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (name, flagged) in enumerate([('download', flagged_download), ('upload', flagged_upload)]):
    color = BLUE if name == 'download' else ORANGE
    for col, (field, label) in enumerate([('client_as_name', 'ASN'), ('state', 'state'), ('date', 'date')]):
        counts = flagged[field].fillna('<missing>').value_counts().head(8).sort_values()
        ax = axes[row, col]
        ax.barh(counts.index, counts.values, color=color)
        ax.set_title(f'{name.title()}: top {label} among tail rows (n={len(flagged)})', fontsize=10)
        ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'tail_diversity.png'), dpi=200, bbox_inches='tight'); plt.show()

print(f'Download tail: {flagged_download["client_as_name"].nunique()} distinct ASNs, '
      f'{flagged_download["state"].nunique()} distinct states, {flagged_download["date"].nunique()} distinct dates '
      f'across {len(flagged_download)} rows')
print(f'Upload tail:   {flagged_upload["client_as_name"].nunique()} distinct ASNs, '
      f'{flagged_upload["state"].nunique()} distinct states, {flagged_upload["date"].nunique()} distinct dates '
      f'across {len(flagged_upload)} rows')
print('Wide spread across ASN/state/date = looks like many independent bad links, not one bad batch.')

In [ ]:
def tail_boxplots(df, flag, features, dataset):
    fig, axes = plt.subplots(1, len(features), figsize=(4.6 * len(features), 4.3))
    axes = np.atleast_1d(axes)
    for ax, feature in zip(axes, features):
        rest = df.loc[~flag, feature].dropna()
        tail = df.loc[flag, feature].dropna()
        bp = ax.boxplot([rest, tail], tick_labels=['Rest of cleaned data', 'Tail (review)'],
                        showfliers=False, patch_artist=True, widths=.55)
        for patch, color in zip(bp['boxes'], [GREY, RED]):
            patch.set_facecolor(color); patch.set_alpha(.65)
        if 'loss' not in feature:
            ax.set_yscale('log')
        ax.set_title(f'{dataset.title()}: {feature}', fontsize=10)
        ax.grid(axis='y', alpha=.25)
    fig.suptitle(f'{dataset.title()}: tail rows vs. rest of cleaned data (fliers hidden for scale)')
    fig.tight_layout(); plt.show()

tail_boxplots(clean_stage1_download, download_joint_flag, DOWNLOAD_FEATURES, 'download')
tail_boxplots(clean_stage1_upload, upload_joint_flag, UPLOAD_FEATURES, 'upload')

In [ ]:
download_loss_median = clean_stage1_download['loss_rate'].median()
tail_high_loss_pct = 100 * (flagged_download['loss_rate'] > download_loss_median).mean()
print(f'Download tail: {tail_high_loss_pct:.1f}% of flagged rows ALSO have above-median loss_rate.')
print('High RTT + high loss + low throughput together is an internally consistent poor-path signature')
print('(one feature spiking alone, with the others normal, would be more suspicious of a data glitch).')

top_download = flagged_download.assign(score=download_joint_score.loc[download_joint_flag]).nlargest(8, 'score')
top_download = top_download[['measurement_id', 'date', 'state', 'city', 'client_as_name',
                              'download_mbps', 'min_rtt_ms', 'loss_rate', 'score']].assign(dataset='download')
top_upload = flagged_upload.assign(score=upload_joint_score.loc[upload_joint_flag]).nlargest(8, 'score')
top_upload = top_upload[['measurement_id', 'date', 'state', 'city', 'client_as_name',
                          'upload_mbps', 'min_rtt_ms', 'score']].assign(dataset='upload')
print('Most extreme 8 rows per direction, for a manual eyeball check:')
display(top_download)
display(top_upload)

### Decision: KEEP the statistical tail (finalized)

These `joint_review_rows` sit in the extreme 0.1% of a combined throughput+RTT(+loss) robust-distance
score, pass every domain-validity rule, and are not matched by the exact-ASN exclusion policy. The deep-dive above shows
them spread across many ASNs/states/dates with internally-consistent poor-path signatures (high RTT
+ high loss + low throughput together) — genuine severe/rural/mobile conditions, not a data artifact.
**Decision (confirmed): keep them.** At 0.10% of rows in each direction they are exactly the "poor"
end of the network-quality spectrum this project cares about, so removing them would only throw away
signal. The clean export in Section 10 already reflects this — nothing further is removed.

## 9. Outlier removal so far — percentage by category

Rows matching reviewed exact-ASN non-access rules and domain-invalid/sentinel rows are removed. The
statistical tail is shown for scale but is **kept by decision**, not removed — see above.

In [ ]:
category_rows = []
for name, raw, validity, non_access, joint_flag, clean_n in [
    ('download', download_raw, download_validity, non_access_download, download_joint_flag, len(clean_stage1_download)),
    ('upload', upload_raw, upload_validity, non_access_upload, upload_joint_flag, len(clean_stage1_upload)),
]:
    n = len(raw)
    domain_n = int(validity['domain_invalid'].sum())
    nonacc_n = int(non_access.sum())
    tail_n = int(joint_flag.sum())
    category_rows += [
        {'dataset': name, 'category': 'Exact-ASN non-access policy (removed)', 'rows': nonacc_n, 'percent': 100 * nonacc_n / n},
        {'dataset': name, 'category': 'Domain-invalid / sentinel (removed)', 'rows': domain_n, 'percent': 100 * domain_n / n},
        {'dataset': name, 'category': 'Statistical tail (kept by decision)', 'rows': tail_n, 'percent': 100 * tail_n / n},
    ]
removal_summary = pd.DataFrame(category_rows)
removal_summary.to_csv(OUTPUT_DIR / 'outlier_removal_summary.csv', index=False)

order = ['Exact-ASN non-access policy (removed)', 'Domain-invalid / sentinel (removed)', 'Statistical tail (kept by decision)']
pivot = removal_summary.pivot(index='category', columns='dataset', values='percent').loc[order]
pivot_rows = removal_summary.pivot(index='category', columns='dataset', values='rows').loc[order]
# Label with both percent AND the raw row count -- "0.00%" alone hides the difference between
# 0 rows (download) and 1 row (upload), and between a 27-row and a 33-row tail.
ax = pivot.plot.barh(figsize=(9, 4.5), color=[BLUE, ORANGE])
for container, col in zip(ax.containers, pivot.columns):
    labels = [f'{pct:.2f}% ({int(rows):,} row{"s" if rows != 1 else ""})'
              for pct, rows in zip(pivot[col], pivot_rows[col])]
    ax.bar_label(container, labels=labels, fontsize=8, padding=3)
ax.set(title='Outliers removed so far, by category (% of original rows, with row counts)',
       xlabel='Percent of original rows', ylabel='')
ax.set_xlim(0, pivot.values.max() * 1.4)  # headroom so the longer row-count labels don't get clipped
ax.grid(axis='x', alpha=.25); ax.invert_yaxis()
fig = ax.get_figure(); fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'outlier_removal_by_category.png'), dpi=200, bbox_inches='tight'); plt.show()

print('Removed (data-centre + domain-invalid/sentinel):')
for name in ['download', 'upload']:
    pct = removal_summary.loc[(removal_summary['dataset'] == name) &
                               (removal_summary['category'] != 'Statistical tail (kept by decision)'), 'percent'].sum()
    print(f'  {name}: {pct:.2f}% removed, {100 - pct:.2f}% retained')
print('Statistical tail — kept by decision (counted in "retained" above, shown separately for scale):')
for name in ['download', 'upload']:
    pct = removal_summary.loc[(removal_summary['dataset'] == name) &
                               (removal_summary['category'] == 'Statistical tail (kept by decision)'), 'percent'].iloc[0]
    print(f'  {name}: {pct:.2f}%')

In [ ]:
# Same numbers as the chart above, as an actual table -- easier to drop straight into a slide
# or read the exact figures off of, instead of estimating bar lengths.
table = removal_summary.pivot(index='category', columns='dataset', values=['rows', 'percent']).loc[order]
table.columns = [f'{ds}_{stat}' for stat, ds in table.columns]
table = table[['download_rows', 'download_percent', 'upload_rows', 'upload_percent']].reset_index()
table.columns = ['Category', 'Download rows', 'Download %', 'Upload rows', 'Upload %']
table['Status'] = ['Removed', 'Removed', 'Kept (decision)']
table.to_csv(OUTPUT_DIR / 'outlier_removal_table.csv', index=False)
display(table.round(2))

fig, ax = plt.subplots(figsize=(13, 2.3))
ax.axis('off')
cell_text = table.copy()
cell_text['Download rows'] = cell_text['Download rows'].map('{:,.0f}'.format)
cell_text['Upload rows'] = cell_text['Upload rows'].map('{:,.0f}'.format)
cell_text['Download %'] = cell_text['Download %'].map('{:.2f}%'.format)
cell_text['Upload %'] = cell_text['Upload %'].map('{:.2f}%'.format)
tbl = ax.table(cellText=cell_text.values, colLabels=cell_text.columns, cellLoc='center', loc='center',
               colWidths=[.34, .13, .12, .12, .12, .17])
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 2.4)
for i in range(len(cell_text) + 1):
    tbl[(i, 0)].set_text_props(ha='left'); tbl[(i, 0)].PAD = .02
for j in range(len(cell_text.columns)):
    tbl[(0, j)].set_facecolor(BLUE); tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for i in range(1, len(cell_text) + 1):
    row_color = '#FDECEA' if cell_text.loc[i - 1, 'Status'] == 'Removed' else '#EAF4EA'
    for j in range(len(cell_text.columns)):
        tbl[(i, j)].set_facecolor(row_color)
fig.suptitle('Outlier removal summary — exactly what was removed vs. kept', y=.98, fontsize=12)
fig.tight_layout(); plt.show()

## 10. Clean dataset export

Funnel: original rows → minus domain-invalid (incl. sentinel) → minus exact-ASN non-access matches →
**clean dataset exported below**. The statistical-tail rows from Section 8 are included by
confirmed decision (kept, not an artifact).

In [ ]:
funnel = pd.DataFrame([
    {'dataset': 'download', 'stage': '1. Original', 'rows': len(download_raw)},
    {'dataset': 'download', 'stage': '2. - domain invalid', 'rows': len(download_raw) - download_validity['domain_invalid'].sum()},
    {'dataset': 'download', 'stage': '3. - exact-ASN non-access matches (clean)', 'rows': len(clean_stage1_download)},
    {'dataset': 'upload', 'stage': '1. Original', 'rows': len(upload_raw)},
    {'dataset': 'upload', 'stage': '2. - domain invalid', 'rows': len(upload_raw) - upload_validity['domain_invalid'].sum()},
    {'dataset': 'upload', 'stage': '3. - exact-ASN non-access matches (clean)', 'rows': len(clean_stage1_upload)},
])
# Each dataset gets its own y-axis, zoomed to that dataset's own row-count range -- on a shared
# 0-33,000 axis the 1-4% drop this project actually makes is nearly invisible (all bars look the
# same height); zooming in is what makes the funnel actually look like a funnel.
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, name, color in zip(axes, ['download', 'upload'], [BLUE, ORANGE]):
    sub = funnel.loc[funnel['dataset'] == name].reset_index(drop=True)
    bars = ax.bar(sub['stage'], sub['rows'], color=color, width=.55)
    ax.bar_label(bars, fmt='{:,.0f}', padding=3)
    for i in range(1, len(sub)):
        removed = sub.loc[i - 1, 'rows'] - sub.loc[i, 'rows']
        if removed > 0:
            ax.annotate(f'\u2212{removed:,} rows', xy=(i - .5, (sub.loc[i - 1, 'rows'] + sub.loc[i, 'rows']) / 2),
                        ha='center', va='center', fontsize=9, color=RED, fontweight='bold')
    ax.set_ylim(sub['rows'].min() * 0.985, sub['rows'].max() * 1.015)
    ax.set_xticks(range(len(sub))); ax.set_xticklabels(sub['stage'], rotation=15, ha='right')
    ax.set(title=f'{name.title()}: row-count funnel (y-axis zoomed to show the drop)', ylabel='Rows')
    ax.grid(axis='y', alpha=.25)
fig.suptitle('Original -> minus domain-invalid -> minus exact-ASN policy matches = clean dataset')
fig.tight_layout(); fig.savefig(figure_path('02_cleaning_evidence', 'cleaning_funnel.png'), dpi=200, bbox_inches='tight'); plt.show()

clean_stage1_download.to_csv(OUTPUT_DIR / 'download_clean.csv', index=False)
clean_stage1_upload.to_csv(OUTPUT_DIR / 'upload_clean.csv', index=False)

def build_removed(df_raw, validity, non_access_flag):
    non_access_aligned = non_access_flag.reindex(df_raw.index, fill_value=False)
    reason = pd.Series('', index=df_raw.index, dtype='object')
    reason.loc[non_access_aligned] = 'non_access_asn'
    reason.loc[validity['domain_invalid']] = 'domain_invalid_sentinel'  # higher priority label if both
    removed_mask = validity['domain_invalid'] | non_access_aligned
    removed = df_raw.loc[removed_mask].copy()
    removed.insert(1, 'removal_reason', reason.loc[removed_mask])
    return removed

removed_download = build_removed(download_raw, download_validity, non_access_download)
removed_upload = build_removed(upload_raw, upload_validity, non_access_upload)
removed_download.to_csv(OUTPUT_DIR / 'download_removed_outliers.csv', index=False)
removed_upload.to_csv(OUTPUT_DIR / 'upload_removed_outliers.csv', index=False)
print(f'Removed rows saved: {len(removed_download)} download, {len(removed_upload)} upload '
      f'-> download_removed_outliers.csv / upload_removed_outliers.csv')

summary = {
    'download_original_rows': int(len(download_raw)),
    'download_domain_invalid_removed': int(download_validity['domain_invalid'].sum()),
    'download_non_access_removed': int(non_access_download.sum()),
    'download_clean_rows': int(len(clean_stage1_download)),
    'download_statistical_tail_kept_by_decision': int(download_joint_flag.sum()),
    'upload_original_rows': int(len(upload_raw)),
    'upload_domain_invalid_removed': int(upload_validity['domain_invalid'].sum()),
    'upload_non_access_removed': int(non_access_upload.sum()),
    'upload_clean_rows': int(len(clean_stage1_upload)),
    'upload_statistical_tail_kept_by_decision': int(upload_joint_flag.sum()),
}
with (OUTPUT_DIR / 'cleaning_summary.json').open('w') as fh:
    json.dump(summary, fh, indent=2)
print(json.dumps(summary, indent=2))
print('\nSaved: download_clean.csv, upload_clean.csv')
print('Statistical-tail rows are included by confirmed decision (see Section 8a).')

## 9a. Outlier review — exactly what was removed (for the presentation)

The relevant cleaning figures are grouped in `figures/02_cleaning_evidence/` so it's
obvious what went into the "clean" dataset without digging through every section: the funnel, the
by-category breakdown, and the top removed organizations. Also below: ASNs whose name *sounded*
suspicious (enterprise/institutional/brand names that could be misread) but were individually
checked against their actual traffic (RTT, geographic spread) and **explicitly kept** — most
notably "Skynet", which is just the brand name of three small Indian regional ISPs here
(Skynet India Internet Services, Skynet Broadband Plus Solution, Skynet Datacom Pvt Ltd), not a
data-centre or military network. No military/defense network was found or removed anywhere in
this dataset — every removed ASN is cloud/CDN/hosting, proxy/security, enterprise, civilian
government/institutional, or academic (see `non_access_asn_rules.csv` for the exact list).

In [ ]:

REVIEWED_OUTPUT = OUTPUT_DIR / 'reviewed_but_kept.csv'

REVIEWED_NOT_REMOVED = [
    {'client_asn': 9829, 'name': 'National Internet Backbone (BSNL)',
     'why_kept': 'Ordinary broadband customers: 1,321 rows, median RTT 22.6ms, spread across 32 states/191 cities.'},
    {'client_asn': 24186, 'name': 'RailTel Corporation',
     'why_kept': '319 rows, median RTT 24.6ms, 12 states/22 cities -- same access-network pattern as BSNL.'},
    {'client_asn': 132559, 'name': 'Gatik Business Solutions',
     'why_kept': 'Single-city small ISP, normal RTT.'},
    {'client_asn': 18002, 'name': '(generic ASN, no enterprise signal)',
     'why_kept': 'Single-city small ISP, normal RTT.'},
    {'client_asn': 132570, 'name': 'Skynet India Internet Services',
     'why_kept': 'Just an ISP brand name, not a data-centre or military network -- 1 row, ordinary ISP-range RTT.'},
    {'client_asn': 132566, 'name': 'Skynet Broadband Plus Solution', 'why_kept': 'Same as above.'},
    {'client_asn': 137157, 'name': 'Skynet Datacom Pvt Ltd', 'why_kept': 'Same as above.'},
    {'client_asn': 132972, 'name': '171 Incorporation',
     'why_kept': 'Small regional ISP, 11 rows all in Bhagalpur (Bihar), RTT 56-114ms -- exactly the '
                 'underserved-area population this dataset should keep.'},
]
reviewed_not_removed = pd.DataFrame(REVIEWED_NOT_REMOVED)
reviewed_not_removed.to_csv(REVIEWED_OUTPUT, index=False)
display(reviewed_not_removed)

print('No military or defense network was found or removed anywhere in this dataset.')
print(f'{len(NON_ACCESS_RULES)} reviewed exact-ASN rules were applied across data-centre/cloud/CDN/proxy/'
      f'enterprise/government-institutional/academic and named Wi-Fi-service networks -- see non_access_asn_rules.csv for the '
      f'full removed list.')
print(f'Reviewed-but-kept ASN list: {REVIEWED_OUTPUT}')

## 11. Where the removed outliers came from (geo)

Removed rows are colour-coded by reason. The sentinel row is a single point (upload only). The
remaining marked points show IP geolocation only; neither their location nor ASN name proves the
physical test location or access type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
reason_colors = {'non_access_asn': RED, 'domain_invalid_sentinel': '#F2B701'}
for ax, kept, removed, name in [
    (axes[0], clean_stage1_download, removed_download, 'download'),
    (axes[1], clean_stage1_upload, removed_upload, 'upload')]:
    ax.scatter(kept['longitude'], kept['latitude'], s=3, alpha=.06, color=GREY, label=f'Clean (n={len(kept):,})')
    for reason, color in reason_colors.items():
        subset = removed.loc[removed['removal_reason'] == reason]
        if len(subset):
            ax.scatter(subset['longitude'], subset['latitude'], s=26, alpha=.8, color=color,
                       label=f'{reason} (n={len(subset)})', edgecolors='black', linewidths=.3)
    ax.set(xlim=(67, 99), ylim=(6, 38), title=f'{name.title()}: removed rows by reason', xlabel='Longitude', ylabel='Latitude')
    ax.grid(alpha=.15); ax.legend(fontsize=7, loc='lower left')
fig.suptitle('IP-geolocation coordinates, not verified facility addresses')
fig.tight_layout(); plt.show()

## 12. Before vs. after: effect of cleaning on summary statistics

Mean, median and standard deviation for each feature, original data vs. the cleaned export. Rows
are flagged by exact-ASN review, not by these metric values; this comparison describes the effect
of the policy and is not independent proof of each network's access type.

In [ ]:
def compare_stats(raw_df, clean_df, features):
    rows = []
    for f in features:
        o, c = raw_df[f], clean_df[f]
        rows.append({'feature': f, 'mean_original': o.mean(), 'mean_clean': c.mean(),
                     'median_original': o.median(), 'median_clean': c.median(),
                     'std_original': o.std(), 'std_clean': c.std(),
                     'p99_original': o.quantile(.99), 'p99_clean': c.quantile(.99)})
    return pd.DataFrame(rows)

download_compare = compare_stats(download_raw, clean_stage1_download, DOWNLOAD_FEATURES)
upload_compare = compare_stats(upload_raw, clean_stage1_upload, UPLOAD_FEATURES)
download_compare.to_csv(OUTPUT_DIR / 'download_stats_comparison.csv', index=False)
upload_compare.to_csv(OUTPUT_DIR / 'upload_stats_comparison.csv', index=False)
display(download_compare.round(4))
display(upload_compare.round(4))

for name, cmp_df in [('download', download_compare), ('upload', upload_compare)]:
    pct_change = pd.DataFrame({
        'feature': cmp_df['feature'],
        'mean % change': 100 * (cmp_df['mean_clean'] - cmp_df['mean_original']) / cmp_df['mean_original'],
        'median % change': 100 * (cmp_df['median_clean'] - cmp_df['median_original']) / cmp_df['median_original'],
        'std % change': 100 * (cmp_df['std_clean'] - cmp_df['std_original']) / cmp_df['std_original'],
    }).set_index('feature')
    ax = pct_change.plot.barh(figsize=(9, 1.6 * len(cmp_df) + 1.5), color=[BLUE, ORANGE, GREEN])
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', fontsize=8)
    ax.axvline(0, color='black', lw=.8)
    ax.set(title=f'{name.title()}: effect of cleaning on mean / median / std (% change vs. original)',
           xlabel='% change, clean vs. original')
    ax.grid(axis='x', alpha=.25)
    fig = ax.get_figure(); fig.tight_layout()
    fig.savefig(figure_path('02_cleaning_evidence', f'{name}_cleaning_stat_shift.png'), dpi=200, bbox_inches='tight'); plt.show()

### 12a. Descriptive statistics table — original data, then clean data

Mean, median, mode, standard deviation, min and max for each clustering feature. Original (raw)
data first, clean data second, so you can read either one directly instead of only the % change.

In [ ]:
def descriptive_table(df, features):
    rows = []
    for f in features:
        s = df[f].dropna()
        mode_values = s.mode()
        rows.append({
            'feature': f,
            'count': int(s.count()),
            'mean': s.mean(),
            'median': s.median(),
            'mode': mode_values.iloc[0] if len(mode_values) else np.nan,
            'std': s.std(),
            'min': s.min(),
            'max': s.max(),
        })
    return pd.DataFrame(rows).set_index('feature')

download_stats_original = descriptive_table(download_raw, DOWNLOAD_FEATURES)
download_stats_clean = descriptive_table(clean_stage1_download, DOWNLOAD_FEATURES)
upload_stats_original = descriptive_table(upload_raw, UPLOAD_FEATURES)
upload_stats_clean = descriptive_table(clean_stage1_upload, UPLOAD_FEATURES)

download_stats_original.round(6).to_csv(OUTPUT_DIR / 'download_descriptive_stats_original.csv')
download_stats_clean.round(6).to_csv(OUTPUT_DIR / 'download_descriptive_stats_clean.csv')
upload_stats_original.round(6).to_csv(OUTPUT_DIR / 'upload_descriptive_stats_original.csv')
upload_stats_clean.round(6).to_csv(OUTPUT_DIR / 'upload_descriptive_stats_clean.csv')

print('DOWNLOAD -- original (uncleaned) data')
display(download_stats_original.round(4))
print('DOWNLOAD -- clean data')
display(download_stats_clean.round(4))
print('UPLOAD -- original (uncleaned) data')
display(upload_stats_original.round(4))
print('UPLOAD -- clean data')
display(upload_stats_clean.round(4))

## 13. Connection type: broadband vs. mobile (from `client_as_name`)

Best-effort split using keyword/ASN signals in the ISP name: `GPRS`/`Cellular`/`Vodafone` -> Mobile;
`Broadband`/`Fibernet`/`Telemedia`/`Cable`/etc. -> Broadband. Jio and BSNL are kept as an explicit
**Mixed** bucket rather than guessed, since both route mobile and fixed (JioFiber / BSNL FTTH)
customers under the same corporate ASN with nothing in this dataset to tell them apart — forcing a
guess there would mis-label roughly a quarter to two-fifths of the rows. The mobile vs. broadband
throughput gap below is a sanity check: mobile should be visibly slower, especially on upload.

In [ ]:
CONNECTION_TYPE_OVERRIDES = {
    55836: 'Mixed (Jio: mobile + JioFiber, same ASN)',
    9829: 'Mixed (BSNL backbone: landline/FTTH + possible mobile)',
    9498: 'Mixed (plain "Airtel Ltd", not GPRS or Telemedia)',
    24186: 'Broadband (RailTel/RailWire)',
    4755: 'Enterprise/Wholesale ISP (Tata Comm/VSNL)',
}

def classify_connection(asn, name):
    if asn in CONNECTION_TYPE_OVERRIDES:
        return CONNECTION_TYPE_OVERRIDES[asn]
    n = str(name).lower()
    if any(k in n for k in ['gprs', 'cellular', 'vodafone']):
        return 'Mobile'
    if any(k in n for k in ['broadband', 'fibernet', 'fiber', 'telemedia', 'cable', 'netcom',
                             'internet service provider', 'teleservices isp']):
        return 'Broadband'
    return 'Unclassified (name gives no signal)'

CONNECTION_TYPE_ORDER = ['Mobile', 'Broadband', 'Broadband (RailTel/RailWire)',
    'Mixed (Jio: mobile + JioFiber, same ASN)',
    'Mixed (BSNL backbone: landline/FTTH + possible mobile)',
    'Mixed (plain "Airtel Ltd", not GPRS or Telemedia)',
    'Enterprise/Wholesale ISP (Tata Comm/VSNL)', 'Unclassified (name gives no signal)']
CONNECTION_TYPE_COLORS = {'Mobile': RED, 'Broadband': GREEN, 'Broadband (RailTel/RailWire)': '#8FCB84',
    'Mixed (Jio: mobile + JioFiber, same ASN)': ORANGE,
    'Mixed (BSNL backbone: landline/FTTH + possible mobile)': '#FBB258',
    'Mixed (plain "Airtel Ltd", not GPRS or Telemedia)': '#FDD3A0',
    'Enterprise/Wholesale ISP (Tata Comm/VSNL)': '#9D9D9D',
    'Unclassified (name gives no signal)': GREY}

connection_tables = {}
for name, df, col in [('download', clean_stage1_download, 'download_mbps'),
                       ('upload', clean_stage1_upload, 'upload_mbps')]:
    asn = pd.to_numeric(df['client_asn'], errors='coerce')
    conn_type = pd.Series([classify_connection(a, n) for a, n in zip(asn, df['client_as_name'])], index=df.index)
    counts = conn_type.value_counts().reindex(CONNECTION_TYPE_ORDER).fillna(0).astype(int)
    table = pd.DataFrame({
        'rows': counts,
        'percent': (100 * counts / len(df)).round(2),
        f'median_{col}': [df.loc[conn_type == c, col].median() if counts[c] else np.nan for c in CONNECTION_TYPE_ORDER],
    })
    connection_tables[name] = table
    table.to_csv(OUTPUT_DIR / f'{name}_connection_type.csv')
    print(f'{name.upper()} -- connection type breakdown')
    display(table.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, name in zip(axes, ['download', 'upload']):
    table = connection_tables[name]
    bars = ax.barh(CONNECTION_TYPE_ORDER, table['percent'], color=[CONNECTION_TYPE_COLORS[c] for c in CONNECTION_TYPE_ORDER])
    for b, pct in zip(bars, table['percent']):
        ax.text(pct + 0.5, b.get_y() + b.get_height() / 2, f'{pct:.1f}%', va='center', fontsize=8)
    ax.set(title=f'{name.title()}: connection type (by client_as_name)', xlabel='% of clean rows')
    ax.grid(axis='x', alpha=.25)
fig.tight_layout(); fig.savefig(figure_path('04_connection_type', 'connection_type_split.png'), dpi=180, bbox_inches='tight'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, name, df, col in [(axes[0], 'download', clean_stage1_download, 'download_mbps'),
                           (axes[1], 'upload', clean_stage1_upload, 'upload_mbps')]:
    asn = pd.to_numeric(df['client_asn'], errors='coerce')
    conn_type = pd.Series([classify_connection(a, n) for a, n in zip(asn, df['client_as_name'])], index=df.index)
    simple = conn_type.map(lambda c: 'Mobile' if c == 'Mobile' else ('Broadband' if 'Broadband' in c and 'Mixed' not in c else 'Mixed/Unclassified'))
    med = df.groupby(simple)[col].median().reindex(['Mobile', 'Broadband', 'Mixed/Unclassified'])
    ax.bar(med.index, med.values, color=[RED, GREEN, GREY])
    for i, v in enumerate(med.values):
        ax.text(i, v, f'{v:.1f}', ha='center', va='bottom', fontsize=9)
    ax.set(title=f'{name.title()}: median {col} by type', ylabel=f'Median {col}')
    ax.grid(axis='y', alpha=.25)
fig.tight_layout(); fig.savefig(figure_path('04_connection_type', 'connection_type_speed.png'), dpi=180, bbox_inches='tight'); plt.show()

### 13a. Mobile vs. broadband — clean-data summary table

The charts above split connection type into 8 detailed buckets; here they're collapsed to the
same three groups as the median-throughput chart (Mobile / Broadband / Mixed-Unclassified) with
every useful metric in one place -- row count, share of the clean data, and median throughput,
RTT, and (download only) loss -- for a direct, presentation-ready comparison.

In [ ]:
type_summary_rows = []
for direction, df, col, loss_col in [('Download', clean_stage1_download, 'download_mbps', 'loss_rate'),
                                      ('Upload', clean_stage1_upload, 'upload_mbps', None)]:
    asn = pd.to_numeric(df['client_asn'], errors='coerce')
    conn_type = pd.Series([classify_connection(a, n) for a, n in zip(asn, df['client_as_name'])], index=df.index)
    simple = conn_type.map(lambda c: 'Mobile' if c == 'Mobile'
                            else ('Broadband' if 'Broadband' in c and 'Mixed' not in c else 'Mixed/Unclassified'))
    for t in ['Mobile', 'Broadband', 'Mixed/Unclassified']:
        sub = df.loc[simple == t]
        row = {'Direction': direction, 'Connection type': t, 'Rows': len(sub),
               '% of clean rows': round(100 * len(sub) / len(df), 2),
               'Median throughput (Mbps)': round(sub[col].median(), 2) if len(sub) else np.nan,
               'Median RTT (ms)': round(sub['min_rtt_ms'].median(), 2) if len(sub) else np.nan}
        if loss_col:
            row['Median loss (%)'] = round(100 * sub[loss_col].median(), 2) if len(sub) else np.nan
        type_summary_rows.append(row)
connection_type_summary = pd.DataFrame(type_summary_rows)
connection_type_summary.to_csv(OUTPUT_DIR / 'connection_type_summary_table.csv', index=False)
display(connection_type_summary)

cell_text = connection_type_summary.fillna('\u2014').copy()
fig, ax = plt.subplots(figsize=(13, 0.5 * len(cell_text) + 1.2))
ax.axis('off')
tbl = ax.table(cellText=cell_text.values, colLabels=cell_text.columns, cellLoc='center', loc='center',
               colWidths=[.13, .17, .10, .13, .17, .13, .12])
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 2.1)
type_colors = {'Mobile': '#FDECEA', 'Broadband': '#EAF4EA', 'Mixed/Unclassified': '#F1F1F1'}
for j in range(len(cell_text.columns)):
    tbl[(0, j)].set_facecolor(BLUE); tbl[(0, j)].set_text_props(color='white', fontweight='bold')
for i, conn_t in enumerate(cell_text['Connection type'], start=1):
    for j in range(len(cell_text.columns)):
        tbl[(i, j)].set_facecolor(type_colors[conn_t])
    tbl[(i, 1)].set_text_props(ha='left')
fig.suptitle('Mobile vs. broadband vs. mixed/unclassified — clean data, both directions', y=.98, fontsize=12)
fig.tight_layout(); plt.show()